# Model Dataset Creation

Joins all processed feature tables, applies feature selection, and produces a single
modelling dataset for the Home Credit default-prediction pipeline.

**Pipeline overview**
1. Load all processed parquet feature tables
2. Join into a single customer-level DataFrame
3. Load feature-selection scores & apply selection threshold
4. Diagnostics — duplicate columns / duplicate feature names
5. Cast categorical columns
6. Save the final model dataset

**Output:** `src/intermediate/model_dataset.parquet`

## 1. Imports & Configuration

In [1]:
import os
import pandas as pd
import numpy as np
import gc; 

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")
SRC_ROOT  = os.environ.get("HOMECREDIT_SRC_DIR", r"a:\Dominic\My Projects\Home Credit\src")

VAR_SELECTION_DIR = os.path.join(DATA_ROOT, "Variable-Selection")
STG_DIR = os.path.join(DATA_ROOT, "Staging")
MDS_DIR = os.path.join(DATA_ROOT, "ModelDataset")

# Loan-level metadata table — not a customer-level feature table, excluded from the join
EXCLUDE_FROM_JOIN = {"prev_loan_data_processed.parquet"}

# Feature-selection score file produced by select_vars_refined.ipynb
PATH_SELECTION_METRICS = os.path.join(VAR_SELECTION_DIR, "numeric_selection_metrics_full_df.xlsx")
os.makedirs(VAR_SELECTION_DIR, exist_ok=True)

# Minimum number of passed checks (out of 6 — see select_vars notebook) for a feature
# to be included in the model dataset. Score of 1 or fewer is treated as noise.
MIN_SELECTION_POINTS = 1

PATH_OUTPUT_DATASET  = os.path.join(MDS_DIR, "model_dataset_initial.parquet")
PATH_OUTPUT_FEATURES = os.path.join(MDS_DIR, "selected_features.csv")

## 2. Load & Join Feature Tables

Loads every processed parquet under `src/intermediate/` except the loan-level
`prev_loan_data_processed.parquet` (which is metadata consumed by other notebooks,
not a customer-level feature table) and left-joins them all on `SK_ID_CURR`.

> **Fix:** the original used `files.remove('prev_loan_data_processed.parquet')`,
> which raises `ValueError` if that file is ever renamed or absent. Replaced with a
> set-difference filter that simply skips it if not present.

In [2]:
def downcast_numeric(df, label=""):
    """
    Downcast float64 -> float32 and int64 -> smallest safe int dtype to roughly
    halve memory footprint before merging. SK_ID_CURR is left untouched so join
    correctness (and dtype-matching across tables) is unaffected.
    """
    float_cols = df.select_dtypes(include='float64').columns
    int_cols   = df.select_dtypes(include='int64').columns.difference(['SK_ID_CURR'])

    before_mb = df.memory_usage(deep=True).sum() / 1e6
    if len(float_cols) > 0:
        df[float_cols] = df[float_cols].astype('float32')
    if len(int_cols) > 0:
        df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='integer')
    after_mb = df.memory_usage(deep=True).sum() / 1e6

    print(f"  {label}: {before_mb:,.0f} MB -> {after_mb:,.0f} MB "
          f"({len(float_cols)} float cols, {len(int_cols)} int cols downcast)")
    return df


all_parquet_files = [f for f in os.listdir(STG_DIR) if f.endswith('.parquet')]
# FIX: filter instead of .remove() — never raises if the file is absent
feature_table_files = [f for f in all_parquet_files if f not in EXCLUDE_FROM_JOIN]

# FIX (MemoryError): downcast dtypes immediately on load, before any merge happens.
dataframes = {}
for parquet_file in feature_table_files:
    file_path = os.path.join(STG_DIR, parquet_file)
    key = os.path.splitext(parquet_file)[0]
    df = pd.read_parquet(file_path)
    dataframes[key] = downcast_numeric(df, label=key)
    print(f"Loaded {key}: {dataframes[key].shape}")

  application_train_data_processed: 488 MB -> 297 MB (78 float cols, 44 int cols downcast)
Loaded application_train_data_processed: (307511, 133)
  bureau_processed: 312 MB -> 148 MB (121 float cols, 20 int cols downcast)
Loaded bureau_processed: (263491, 142)
  creditcard_processed: 746 MB -> 361 MB (995 float cols, 71 int cols downcast)
Loaded creditcard_processed: (86905, 1067)
  payment_installments_processed: 329 MB -> 160 MB (118 float cols, 16 int cols downcast)
Loaded payment_installments_processed: (291643, 135)
  posdata_processed: 109 MB -> 50 MB (25 float cols, 15 int cols downcast)
Loaded posdata_processed: (289444, 41)
  prev_loans_aggregate_final_processed: 151 MB -> 74 MB (47 float cols, 11 int cols downcast)
Loaded prev_loans_aggregate_final_processed: (291057, 59)


In [3]:
# Sanity check: every table must have unique SK_ID_CURR, or the left-join below
# would silently fan out rows (row-count explosion).
for name, df in dataframes.items():
    n_dupes = df['SK_ID_CURR'].duplicated().sum()
    assert n_dupes == 0, f"{name} has {n_dupes} duplicate SK_ID_CURR — fix upstream before joining."
print("All tables have unique SK_ID_CURR — safe to join.")

All tables have unique SK_ID_CURR — safe to join.


In [4]:
join_keys = ['SK_ID_CURR']
print(f"Using join key(s): {join_keys}")

combined_df = None

for name, df in dataframes.items():
    df = df.set_index('SK_ID_CURR')

    if combined_df is None:
        combined_df = df
    else:
        # FIX: .join() on a matching index is lighter than repeated .merge()
        # calls keyed on a column — avoids re-hashing SK_ID_CURR every iteration
        combined_df = combined_df.join(df, how='left')

    del df
    gc.collect()   # FIX: reclaim memory after EVERY join, not just at the end
    print(f"After merging {name}: {combined_df.shape}")

combined_df = combined_df.reset_index()

print(f"\nFinal combined DataFrame shape: {combined_df.shape}")

Using join key(s): ['SK_ID_CURR']
After merging application_train_data_processed: (307511, 132)
After merging bureau_processed: (307511, 273)
After merging creditcard_processed: (307511, 1339)
After merging payment_installments_processed: (307511, 1473)
After merging posdata_processed: (307511, 1513)
After merging prev_loans_aggregate_final_processed: (307511, 1571)

Final combined DataFrame shape: (307511, 1572)


In [5]:
del dataframes
gc.collect()

0

## 3. Apply Feature Selection

Loads the per-feature scores produced by `select_vars_refined.ipynb` and keeps any
feature scoring **more than** `MIN_SELECTION_POINTS` (out of 6 checks). All
categorical (`object`-dtype) columns are always retained regardless of score, since
the selection metrics in that notebook are computed on numeric features.

In [6]:
numeric_selection_metrics_full_df = pd.read_excel(PATH_SELECTION_METRICS)

numeric_feature_cols = (
    numeric_selection_metrics_full_df
    .loc[numeric_selection_metrics_full_df['points'] > MIN_SELECTION_POINTS, 'variable']
    .tolist()
)

cat_cols = combined_df.select_dtypes(include=['object', 'category']).columns.tolist()
time_indicator_cols = [c for c in combined_df.columns  if 'IS_IN_' in c]
#cat_cols.remove('SK_ID_CURR')

feature_cols = pd.unique(numeric_feature_cols + cat_cols + time_indicator_cols).tolist()

print(f"Numeric features passing threshold (points > {MIN_SELECTION_POINTS}): {len(numeric_feature_cols)}")
print(f"Categorical features (always retained)                         : {len(cat_cols)}")
print(f"Time indicator features                                        : {len(time_indicator_cols)}")
print(f"Total unique features selected                                 : {len(feature_cols)}")

Numeric features passing threshold (points > 1): 553
Categorical features (always retained)                         : 10
Time indicator features                                        : 27
Total unique features selected                                 : 590


C:\Users\domjo\AppData\Local\Temp\ipykernel_25776\3492365585.py:13: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  feature_cols = pd.unique(numeric_feature_cols + cat_cols + time_indicator_cols).tolist()


In [7]:
pd.Series(feature_cols).to_csv(PATH_OUTPUT_FEATURES, index=False)
print(f"Saved selected feature list → {PATH_OUTPUT_FEATURES}")

Saved selected feature list → a:\Dominic\My Projects\Home Credit\data\ModelDataset\selected_features.csv


## 4. Diagnostics: Duplicate Columns

Verifies the merge in Section 2 did not create duplicate column names, and that
`feature_cols` contains no accidental duplicates.

In [8]:
dup_cols = combined_df.columns[combined_df.columns.duplicated()].unique().tolist()
if dup_cols:
    for col in dup_cols:
        matches = [c for c in combined_df.columns if c == col]
        print(f"  {col}: {len(matches)} copies")
    raise ValueError(f"Duplicate columns found in combined_df: {dup_cols}")
else:
    print("No duplicated combined_df columns detected.")

feature_dup_mask = pd.Series(feature_cols).duplicated(keep=False)
duped_features = pd.Series(feature_cols)[feature_dup_mask].unique().tolist()
if duped_features:
    raise ValueError(f"Duplicate feature names in feature_cols: {duped_features}")
else:
    print("No duplicated feature names in feature_cols.")

No duplicated combined_df columns detected.
No duplicated feature names in feature_cols.


In [9]:
sum(["_x" in col or "_y" in col for col in combined_df.columns if col not in join_keys])

0

## 5. Cast Categorical Columns

> **Note:** category sets are derived from the full dataset here (before any
> train/validation split happens downstream). This is standard practice for
> tree-based models like LightGBM — it only fixes the *set of valid categories*,
> not any target-derived statistic, so it carries no target-leakage risk. It does
> mean that categories unseen at this point (e.g. truly new values at inference
> time) will need explicit handling in the serving pipeline.

In [10]:
for col in cat_cols:
    combined_df[col] = combined_df[col].astype('category')

print(f"Cast {len(cat_cols)} columns to category dtype.")

Cast 10 columns to category dtype.


## 6. Quality Checks & Save

Saves `SK_ID_CURR`, all selected features, and `TARGET` to a single parquet file.
Parquet preserves the pandas `category` dtype via the `pyarrow` engine, so
downstream notebooks reading this file will see the same dtypes.

In [11]:
assert 'TARGET' in combined_df.columns, "TARGET column missing from combined_df!"
n_dupes = combined_df['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR in final dataset: {n_dupes}"

print(f"Final dataset shape : {combined_df.shape}")
print(f"Features            : {len(feature_cols)}")
print(f"Target distribution :\n{combined_df['TARGET'].value_counts(normalize=True).to_string()}")

Final dataset shape : (307511, 1572)
Features            : 590
Target distribution :
TARGET
0    0.919271
1    0.080729


In [12]:
model_dataset = combined_df[['SK_ID_CURR'] + feature_cols + ['TARGET']]
model_dataset.to_parquet(PATH_OUTPUT_DATASET, index=False)
print(f"Saved → {PATH_OUTPUT_DATASET}")

Saved → a:\Dominic\My Projects\Home Credit\data\ModelDataset\model_dataset_initial.parquet


## 7. Cleanup

In [13]:
del combined_df, model_dataset
import gc; gc.collect()
print("All DataFrames released.")

All DataFrames released.
